In [15]:
import pandas as pd
import openpyxl


In [16]:
FILE_PATH = "have_fun.xlsx"

wb = openpyxl.load_workbook(FILE_PATH, data_only=True)
all_sheets = wb.sheetnames
print(all_sheets)

['Ëèñò1', 'Лист1', 'ÌÀÈ', 'ýòî', 'ÿ', '!', 'Ëèñò2_ñòðîêîâûå_NaN_âûáðîñû', 'Ëèñò3_ñìåøàííûå_òèïû']


In [17]:
dfs = {}
for sheet in all_sheets:
    df = pd.read_excel(FILE_PATH, sheet_name=sheet, engine='openpyxl')
    dfs[sheet] = df
    print(
        f" Лист: {sheet} | Shape: {df.shape} | Columns: {list(df.columns)}")
    print(df.head(3))

 Лист: Ëèñò1 | Shape: (125124, 10) | Columns: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure', 'ds', 'city']
   temperature_2m  relative_humidity_2m  precipitation  rain  snowfall  \
0             3.8                    82            0.0   0.0       0.0   
1            -3.0                    70            0.0   0.0       0.0   
2            -1.3                    86            0.0   0.0       0.0   

   weathercode  wind_speed_10m  surface_pressure                  ds  \
0            3            19.1       1010.000000 2020-01-31 02:00:00   
1            3            24.0        981.900024 2025-02-28 09:00:00   
2            2             8.8       1021.200012 2022-01-03 02:00:00   

           city  
0     Ãåëåíäæèê  
1  Áëàãîâåùåíñê  
2     Ãåëåíäæèê  
 Лист: Лист1 | Shape: (0, 0) | Columns: []
Empty DataFrame
Columns: []
Index: []
 Лист: ÌÀÈ | Shape: (0, 0) | Columns: []
Empty DataFrame
Columns: []

In [18]:
SHEETS = ['Ëèñò1', 'Ëèñò2_ñòðîêîâûå_NaN_âûáðîñû', 'Ëèñò3_ñìåøàííûå_òèïû']
for sheet in all_sheets:
    if sheet in SHEETS:
        df= pd.read_excel(FILE_PATH, sheet_name=sheet)
        print(f"Лист {sheet}:")
        dfs[sheet] = df
        print(df.dtypes)
        print(dfs[sheet].head(3).to_string())
        print()

Лист Ëèñò1:
temperature_2m                 float64
relative_humidity_2m             int64
precipitation                  float64
rain                           float64
snowfall                       float64
weathercode                      int64
wind_speed_10m                 float64
surface_pressure               float64
ds                      datetime64[us]
city                               str
dtype: object
   temperature_2m  relative_humidity_2m  precipitation  rain  snowfall  weathercode  wind_speed_10m  surface_pressure                  ds          city
0             3.8                    82            0.0   0.0       0.0            3            19.1       1010.000000 2020-01-31 02:00:00     Ãåëåíäæèê
1            -3.0                    70            0.0   0.0       0.0            3            24.0        981.900024 2025-02-28 09:00:00  Áëàãîâåùåíñê
2            -1.3                    86            0.0   0.0       0.0            2             8.8       1021.200012 2022-01-03

In [19]:
def fix_cp1251(x):
    return x.encode('latin1').decode('cp1251') if isinstance(x, str) else x


raw_sheets = ['Ëèñò1', 'Ëèñò2_ñòðîêîâûå_NaN_âûáðîñû', 'Ëèñò3_ñìåøàííûå_òèïû']
clean_sheets = [fix_cp1251(s) for s in raw_sheets]

dfs = {}
for raw, clean in zip(raw_sheets, clean_sheets):
    df = pd.read_excel(FILE_PATH, sheet_name=raw)

    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

    df['city'] = df['city'].apply(fix_cp1251)


    cols_to_num = ['temperature_2m', 'relative_humidity_2m', 'precipitation',
                   'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']
    for c in cols_to_num:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    df['ds'] = pd.to_datetime(df['ds'])
    dfs[clean] = df

df_all = pd.concat(dfs.values(), ignore_index=True)
df_all = df_all.sort_values(['city', 'ds']).reset_index(drop=True)

print(f"Shape: {df_all.shape}")
print(f"Города: {df_all['city'].unique()}")
print(f"\nТипы данных:\n{df_all.dtypes}")
print(f"\nКоличество пропусков (NaN):\n{df_all.isna().sum()}")
print(df_all.head())

Shape: (373029, 10)
Города: <StringArray>
[   'БЛАГОВЕЩЕНСК',     'Благовещенс',    'Благовещенск',   'Благовещенскк',
       'ГЕЛЕНДЖИК',        'Геленджи',       'Геленджик',      'Геленджикк',
           'Мосва',          'Москва',         'Находка', 'Санкт-Петербург',
             'Соч',            'Сочи',             'Счи',            'Сычи',
    'благовещенск',       'геленджик',               nan]
Length: 19, dtype: str

Типы данных:
temperature_2m                 float64
relative_humidity_2m           float64
precipitation                  float64
rain                           float64
snowfall                       float64
weathercode                    float64
wind_speed_10m                 float64
surface_pressure               float64
ds                      datetime64[us]
city                               str
dtype: object

Количество пропусков (NaN):
temperature_2m          6778
relative_humidity_2m    2459
precipitation           6157
rain                    2484
snowfa

## Удаляем строки, где нет даты или города, тк их нельзя восстановить

In [20]:
df_all = df_all.dropna(subset=['ds', 'city']).copy()
print(f"После удаления строк без ds/city: {df_all.shape}")

После удаления строк без ds/city: (370541, 10)


### Обрабатываем названия городов: убираем опечатки. По итогу в датафрейме представлены данные по 6 городам: Благовещенск, Геленджик, Москва, Сочи, Находка.

In [21]:
df_all['city_clean'] = df_all['city'].astype(str).str.strip().str.title()

CITY_FIXES = {
    'Благовещенс': 'Благовещенск',
    'Благовещенскк': 'Благовещенск',
    'Геленджи': 'Геленджик',
    'Геленджикк': 'Геленджик',
    'Мосва': 'Москва',
    'Соч': 'Сочи',
    'Счи': 'Сочи',
    'Сычи': 'Сочи',
}
df_all['city_clean'] = df_all['city_clean'].replace(CITY_FIXES)

df_all['city_clean'].unique()

<StringArray>
['Благовещенск', 'Геленджик', 'Москва', 'Находка', 'Санкт-Петербург', 'Сочи']
Length: 6, dtype: str

In [22]:
df_all = df_all.drop(columns=['city']).rename(columns={'city_clean': 'city'})
df_all.info()

<class 'pandas.DataFrame'>
Index: 370541 entries, 0 to 370541
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   temperature_2m        366217 non-null  float64       
 1   relative_humidity_2m  370536 non-null  float64       
 2   precipitation         366839 non-null  float64       
 3   rain                  370541 non-null  float64       
 4   snowfall              370541 non-null  float64       
 5   weathercode           368700 non-null  float64       
 6   wind_speed_10m        367452 non-null  float64       
 7   surface_pressure      370541 non-null  float64       
 8   ds                    370541 non-null  datetime64[us]
 9   city                  370541 non-null  str           
dtypes: datetime64[us](1), float64(8), str(1)
memory usage: 31.1 MB


### Сортируем наши данные

In [23]:
df_all = df_all.sort_values(['city', 'ds']).reset_index(drop=True)

### Убираем пропуски в нашем датасете, для этого исползуем линейную интерполяцию


In [24]:
numeric_cols = ['temperature_2m', 'relative_humidity_2m', 'precipitation',
                'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']

def interpolate_city_group(group):
    group = group.set_index('ds')
    for col in numeric_cols:
        if col in group.columns:
            group[col] = group[col].interpolate(
                method='time', limit_direction='both')
    return group.reset_index()

df_all = df_all.groupby('city', group_keys=False).apply(interpolate_city_group)
print(f"Пропуски после интерполяции:\n{df_all[numeric_cols].isna().sum()}")

Пропуски после интерполяции:
temperature_2m          0
relative_humidity_2m    0
precipitation           0
rain                    0
snowfall                0
weathercode             0
wind_speed_10m          0
surface_pressure        0
dtype: int64


In [27]:
df_all.info()

<class 'pandas.DataFrame'>
Index: 370541 entries, 0 to 61367
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   ds                    370541 non-null  datetime64[us]
 1   temperature_2m        370541 non-null  float64       
 2   relative_humidity_2m  370541 non-null  float64       
 3   precipitation         370541 non-null  float64       
 4   rain                  370541 non-null  float64       
 5   snowfall              370541 non-null  float64       
 6   weathercode           370541 non-null  float64       
 7   wind_speed_10m        370541 non-null  float64       
 8   surface_pressure      370541 non-null  float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 28.3 MB


KeyError: 'city'